In [1]:
import re
import csv

# مسیرها
input_path = r"F:\Thesis\project\2-RAG\raw_laws\Civil Procedure Code\divan-edalat-edari.txt"
output_tsv = r"F:\Thesis\project\2-RAG\raw_laws\Civil Procedure Code\preprocessed_data\divan-edalat-edari_articles.tsv"

LAW_CODE = "divan_edalat_edari"
LAW_NAME = "قانون تشکیلات و آیین‌دادرسی دیوان عدالت اداری"


def normalize_digits(text: str) -> str:
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    return text.translate(str.maketrans(persian_digits, english_digits))


def normalize_persian(text: str) -> str:
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 1) خواندن
with open(input_path, "r", encoding="utf-8") as f:
    raw = f.read()

raw = normalize_digits(raw)
raw = raw.replace("–", "-").replace("—", "-").replace("ـ", "-")

# 2) شکستن خطوط برای متادیتاهای معتبر
patterns_to_break = [
    r"(بخش\s+(نخست|اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|\d+)[^\n]*)",
    r"(فصل\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|سیزدهم|نخست|\d+)[^\n]*)",
    r"(مبحث\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|\d+)[^\n]*)",
    r"((?:ماده|مواد)\s*‌?\s*\d+(?:تا\d+)?(?:\s*و\s*\d+)?(?:مکرر)?(?:\([^)]+\))?\s*[-–ـ][^\n]*)"
]

for p in patterns_to_break:
    raw = re.sub(r"(?<!^)" + p, r"\n\1", raw, flags=re.MULTILINE | re.UNICODE)

lines = raw.splitlines()

# 3) الگوهای دقیق (با شماره ترتیبی + خط تیره اجباری)
section_pattern = re.compile(
    r"^بخش\s+(نخست|اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|\d+)\s*-\s*(.*)$"
)

chapter_pattern = re.compile(
    r"^فصل\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|سیزدهم|چهاردهم|پانزدهم|نخست|\d+)\s*-\s*(.*)$"
)

topic_pattern = re.compile(
    r"^مبحث\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|\d+)\s*-\s*(.*)$"
)

# الگوی جامع ماده
article_pattern = re.compile(
    r"^\s*(?:ماده|مواد)"
    r"(?:\s|‌)*"
    r"("
        r"\d+"
        r"(?:\s*تا\s*\d+)?"
        r"(?:\s*و\s*\d+)*"
    r")"
    r"(?:\s*مکرر)?"
    r"(?:\s*\([^)]+\))?"
    r"\s*"
    r"[-–ـ]"
    r"\s*(.*)$",
    re.UNICODE
)

# متغیرهای سلسله‌مراتب (از بزرگ به کوچک)
current_section = ""  # بخش (بزرگترین)
current_chapter = ""  # فصل (زیرمجموعه بخش)
current_topic = ""    # مبحث (زیرمجموعه فصل)

articles = []
current_article_num = None
current_article_text = ""


def flush_article():
    global current_article_num, current_article_text
    if current_article_num is not None and current_article_text.strip():
        articles.append(
            {
                "law_code": LAW_CODE,
                "law_name": LAW_NAME,
                "section": current_section,
                "chapter": current_chapter,
                "topic": current_topic,
                "article_number": current_article_num,
                "text": normalize_persian(current_article_text),
            }
        )
    current_article_num = None
    current_article_text = ""


# پردازش
for line in lines:
    stripped = line.strip()
    if not stripped:
        continue

    # 1) بخش (بزرگترین سطح)
    m = section_pattern.match(stripped)
    if m:
        flush_article()
        current_section = stripped
        # ریست سطوح پایین‌تر
        current_chapter = ""
        current_topic = ""
        continue

    # 2) فصل (زیرمجموعه بخش)
    m = chapter_pattern.match(stripped)
    if m:
        flush_article()
        current_chapter = stripped
        # ریست سطوح پایین‌تر
        current_topic = ""
        continue

    # 3) مبحث (زیرمجموعه فصل)
    m = topic_pattern.match(stripped)
    if m:
        flush_article()
        current_topic = stripped
        continue

    # 4) ماده
    m = article_pattern.match(stripped)
    if m:
        flush_article()
        num_str = m.group(1)
        rest = m.group(2).strip()

        # استخراج اولین عدد
        first_num = re.search(r"\d+", num_str)
        if first_num:
            try:
                current_article_num = int(first_num.group())
            except ValueError:
                current_article_num = None
        else:
            current_article_num = None

        current_article_text = f"ماده {num_str}- {rest}"
        continue

    # 5) ادامه متن ماده
    if current_article_num is not None:
        current_article_text += " " + stripped

# آخرین ماده
flush_article()

# خروجی TSV
fieldnames = ["law_code", "law_name", "section", "chapter", "topic", "article_number", "text"]

with open(output_tsv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    for row in articles:
        writer.writerow(row)

print("✓ پردازش قانون نحوه اجرای محکومیت‌های مالی تمام شد.")
print(f"✓ تعداد مواد استخراج شده: {len(articles)}")
print(f"✓ خروجی TSV: {output_tsv}")


✓ پردازش قانون نحوه اجرای محکومیت‌های مالی تمام شد.
✓ تعداد مواد استخراج شده: 115
✓ خروجی TSV: F:\Thesis\project\2-RAG\raw_laws\Civil Procedure Code\preprocessed_data\divan-edalat-edari_articles.tsv
